# CSCI 3151 - Foundations of Machine Learning
## M53 - Transformer Encoder-Decoder Architectures

In this mini-lab, you will work with a **small synthetic sequence-classification problem**. Each sequence contains two special tokens, and the label depends only on **which one appears first**. That makes the task **order-sensitive**: a model that ignores position should struggle, while a transformer encoder with positional information should do much better.

**Context / DGP.** We generate short token sequences with simple, controlled rules rather than using a large real NLP corpus. That lets us isolate the architectural ideas from the lecture: positional encodings, self-attention, encoder blocks, masking, and ablation-based reasoning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 3151
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
print('Torch version:', torch.__version__)

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


## Synthetic sequence task

We will use sequences of length `T = 12` over a vocabulary of size `10`.

- Tokens `9` and `10` are the **special tokens**.
- Every sequence contains **exactly one** `9` and **exactly one** `10`.
- The label is `1` if token `9` appears before token `10`, and `0` otherwise.

This is a useful toy problem because the **multiset of tokens is not enough** to solve it. The model needs access to **relative order**.


In [ ]:
T = 12
VOCAB_SIZE = 10
A_TOKEN = 9
B_TOKEN = 10
FILLER_TOKENS = np.arange(1, 9)

def make_order_dataset(n_samples, seq_len=T, seed=SEED):
    rng = np.random.default_rng(seed)
    X = np.zeros((n_samples, seq_len), dtype=np.int64)
    y = np.zeros(n_samples, dtype=np.int64)

    for i in range(n_samples):
        seq = rng.choice(FILLER_TOKENS, size=seq_len, replace=True)
        pos_a, pos_b = rng.choice(seq_len, size=2, replace=False)
        seq[pos_a] = A_TOKEN
        seq[pos_b] = B_TOKEN
        X[i] = seq
        y[i] = int(pos_a < pos_b)

    return X, y

N = 2000
X, y = make_order_dataset(N)

n_train = int(0.70 * N)
n_val = int(0.15 * N)

X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:n_train + n_val], y[n_train:n_train + n_val]
X_test, y_test = X[n_train + n_val:], y[n_train + n_val:]

print('Shapes:')
print('  train:', X_train.shape, y_train.shape)
print('  val:  ', X_val.shape, y_val.shape)
print('  test: ', X_test.shape, y_test.shape)
print()
print('Class balance:')
print('  train positive rate:', round(float(y_train.mean()), 3))
print('  val positive rate:  ', round(float(y_val.mean()), 3))
print('  test positive rate: ', round(float(y_test.mean()), 3))

preview = pd.DataFrame({
    'sequence': [' '.join(map(str, row)) for row in X_train[:5]],
    'label': y_train[:5]
})
preview


## Core Task 0 - Warm-up: decoder masks and architecture choice

Before training anything, connect the lecture ideas to concrete objects.

1. Run the next cell to build and visualize a **causal mask**.
2. In a markdown cell below it, answer:
   - Why does an **encoder-only** classifier *not* need this mask?
   - Why does a **decoder-only** language model *need* it?
   - Which architecture would you choose for:
     1. sentiment classification,
     2. next-token prediction,
     3. English to French translation?


In [ ]:
def causal_mask(seq_len):
    blocked = np.triu(np.ones((seq_len, seq_len), dtype=int), k=1)
    allowed = 1 - blocked
    return allowed

mask = causal_mask(8)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.imshow(mask, cmap='Blues', vmin=0, vmax=1)
ax.set_title('Causal mask (1 = allowed, 0 = blocked)')
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
ax.set_xticks(range(mask.shape[1]))
ax.set_yticks(range(mask.shape[0]))

for i in range(mask.shape[0]):
    for j in range(mask.shape[1]):
        ax.text(j, i, str(mask[i, j]), ha='center', va='center', color='white' if mask[i, j] == 1 else 'black')

plt.tight_layout()
plt.show()


### Your notes for Task 0

Write 2-4 sentences here about **where causal masking belongs** and **why**.


## Core Task 1 - Positional encodings

M53 emphasized that self-attention alone is **permutation-equivariant**.
So we now add **sinusoidal positional encodings** and inspect what they look like.

As you run the next cell, look for two things:
- nearby positions should have **similar** encodings;
- different positions should still get **distinct** vectors.


In [ ]:
def sinusoidal_pe_numpy(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d_model // 2)[None, :]
    denom = 10000 ** (2 * i / d_model)
    pe = np.zeros((seq_len, d_model), dtype=np.float32)
    pe[:, 0::2] = np.sin(pos / denom)
    pe[:, 1::2] = np.cos(pos / denom)
    return pe

d_model_demo = 32
PE = sinusoidal_pe_numpy(T, d_model_demo)
PE_norm = PE / np.linalg.norm(PE, axis=1, keepdims=True)
sim = PE_norm @ PE_norm.T

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

im0 = axes[0].imshow(PE.T, aspect='auto', origin='lower', cmap='coolwarm')
axes[0].set_title('Sinusoidal positional encodings')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Dimension')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(sim, origin='lower', cmap='viridis', vmin=-0.2, vmax=1.0)
axes[1].set_title('Cosine similarity between positions')
axes[1].set_xlabel('Position j')
axes[1].set_ylabel('Position i')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

pd.DataFrame(np.round(sim[:6, :6], 3), index=[f'pos{i}' for i in range(6)], columns=[f'pos{j}' for j in range(6)])


### Quick reflection for Task 1

Add a short answer here:

- What in the plots suggests that **nearby positions are similar**?
- Why is that useful for a transformer?
- Why is similarity **not** the same thing as two positions being identical?


## Core Task 2 - Train a tiny transformer encoder classifier

We now build a small **encoder-only transformer** for this order-sensitive task.

High-level recipe:
1. token embeddings,
2. optional positional encodings,
3. one transformer encoder block,
4. mean pooling over positions,
5. a linear classifier.

This is intentionally the simpler **encoder-only** case that M53 says is appropriate for sequence understanding and classification.


In [ ]:
def make_loaders(X_train, y_train, X_val, y_val, X_test, y_test, batch_size=128):
    X_train_t = torch.tensor(X_train, dtype=torch.long)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    X_val_t = torch.tensor(X_val, dtype=torch.long)
    y_val_t = torch.tensor(y_val, dtype=torch.long)
    X_test_t = torch.tensor(X_test, dtype=torch.long)
    y_test_t = torch.tensor(y_test, dtype=torch.long)
    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders(X_train, y_train, X_val, y_val, X_test, y_test)

def sinusoidal_pe_torch(seq_len, d_model):
    pos = torch.arange(seq_len).unsqueeze(1).float()
    i = torch.arange(d_model // 2).unsqueeze(0).float()
    denom = 10000 ** (2 * i / d_model)
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(pos / denom)
    pe[:, 1::2] = torch.cos(pos / denom)
    return pe

class TinyTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=32, nhead=4, num_layers=1, d_ff=64, max_len=T, num_classes=2, use_positional_encoding=True, dropout=0.0):
        super().__init__()
        self.use_positional_encoding = use_positional_encoding
        self.embedding = nn.Embedding(vocab_size + 1, d_model)
        pe = sinusoidal_pe_torch(max_len, d_model).unsqueeze(0)
        self.register_buffer('pe', pe)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=d_ff, dropout=dropout, batch_first=True, activation='relu')
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        h = self.embedding(x)
        if self.use_positional_encoding:
            h = h + self.pe[:, :x.size(1), :]
        h = self.encoder(h)
        pooled = h.mean(dim=1)
        return self.classifier(pooled)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def accuracy_from_logits(logits, y):
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()

def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    total_acc = 0.0
    total_n = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        if train_mode:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train_mode:
                loss.backward()
                optimizer.step()

        batch_size = xb.size(0)
        total_loss += loss.item() * batch_size
        total_acc += accuracy_from_logits(logits, yb) * batch_size
        total_n += batch_size

    return total_loss / total_n, total_acc / total_n

def train_model(use_positional_encoding=True, nhead=4, num_layers=1, d_model=32, d_ff=64, epochs=6, lr=2e-3):
    torch.manual_seed(SEED)
    model = TinyTransformerClassifier(vocab_size=VOCAB_SIZE, d_model=d_model, nhead=nhead, num_layers=num_layers, d_ff=d_ff, use_positional_encoding=use_positional_encoding).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    rows = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, optimizer=None)
        rows.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc, 'val_loss': val_loss, 'val_acc': val_acc})

    test_loss, test_acc = run_epoch(model, test_loader, optimizer=None)
    history = pd.DataFrame(rows)
    return model, history, test_loss, test_acc

model_pe, hist_pe, test_loss_pe, test_acc_pe = train_model(use_positional_encoding=True, nhead=4, epochs=6)
print('Trainable parameters:', count_parameters(model_pe))
print(hist_pe)
print('Test accuracy with positional encoding:', round(float(test_acc_pe), 3))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist_pe['epoch'], hist_pe['train_acc'], marker='o', label='Train accuracy')
ax.plot(hist_pe['epoch'], hist_pe['val_acc'], marker='s', label='Validation accuracy')
ax.axhline(test_acc_pe, color='black', linestyle='--', linewidth=1.2, label=f'Test acc = {test_acc_pe:.3f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Tiny transformer encoder on an order-sensitive task')
ax.set_ylim(0.45, 1.02)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Reflection for Task 2

Add a short interpretation here:

- Did the model learn the task quickly or slowly?
- Why is this task a good test of **positional information** rather than just token identity?
- What part of the M53 architecture is most directly responsible for the model's success here?


## Core Task 3 - Ablation: what happens if we remove positional encoding?

A good engineering habit is to change **one thing at a time**.

We will compare three small models:
1. **Transformer + positional encoding + 4 heads**
2. **Transformer without positional encoding + 4 heads**
3. **Transformer + positional encoding + 1 head**

You should expect the **no-positional-encoding** model to struggle because this task depends on order.


In [ ]:
configs = [
    ('PE + 4 heads', True, 4),
    ('No PE + 4 heads', False, 4),
    ('PE + 1 head', True, 1)
]

rows = []
for name, use_pe, nhead in configs:
    model_i, hist_i, test_loss_i, test_acc_i = train_model(use_positional_encoding=use_pe, nhead=nhead, epochs=5)
    rows.append({
        'experiment': name,
        'final_train_acc': hist_i['train_acc'].iloc[-1],
        'final_val_acc': hist_i['val_acc'].iloc[-1],
        'test_acc': test_acc_i,
        'n_parameters': count_parameters(model_i)
    })

results = pd.DataFrame(rows).sort_values('test_acc', ascending=False)
results

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(results['experiment'], results['test_acc'])
ax.set_ylabel('Test accuracy')
ax.set_title('Ablation results')
ax.set_ylim(0.0, 1.02)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


### Interpretation for Task 3

Write 3-5 sentences here.

Suggested prompts:
- Which ablation mattered most?
- Did changing the number of heads matter as much as changing positional encoding?
- What does this tell you about the difference between **capacity** and **the right inductive bias**?


## Stretch tasks

Choose **one or more** if you finish early.

### Stretch A - Make the task order-invariant
Create a new dataset where the label is based on something like:
- whether the **sum** of tokens is above a threshold, or
- whether token `9` appears **anywhere** in the sequence.

Then compare **with** and **without** positional encoding. Do you still expect PE to matter as much?

### Stretch B - Depth or head count
Try one of:
- `num_layers = 2`
- `nhead = 2`
- `d_model = 64`

Keep everything else fixed and record whether validation/test accuracy changes enough to justify the extra capacity.

### Stretch C - Architecture matching
In a markdown cell, justify the best transformer family for each task:
- document classification,
- autocomplete,
- machine translation,
- sentence embedding for retrieval.

Use the language of **encoder-only**, **decoder-only**, and **encoder-decoder** from M53.

### Stretch D - Better experimental hygiene
Repeat one experiment with **three different seeds** and report the mean and standard deviation of test accuracy. Does your conclusion still hold?

---

## Submission / wrap-up

Before you submit, make sure your notebook includes:
- a short answer for **Task 0** about masking and architecture choice;
- a short interpretation for **Task 1** about what the positional encoding plots show;
- a short interpretation for **Task 2** about why this is a good test of positional information;
- a short interpretation for **Task 3** about the ablation results;
- any stretch work you chose to complete.

The goal is not just to get a high score on a toy problem, but to connect **architecture diagrams**, **mathematical ideas**, and **empirical evidence**.
